# Data Science avec Pandas : les concepts clés

Notebook de l'article [Data Science avec Pandas](https://moncoachdata.com/blog/data-science-avec-pandas/) de MonCoachData. Données : [Netflix Movies and TV Shows](https://www.kaggle.com/datasets/dgoenrique/netflix-movies-and-tv-shows) (Kaggle). Code testé avec Python 3.13 et Pandas 3.

## Télécharger les données

Les deux fichiers CSV sont téléchargés depuis GitHub (rien n'est téléchargé s'ils sont déjà présents).

In [ ]:
import os
from urllib.request import urlretrieve

for fichier in ("titles.csv", "credits.csv"):
    if not os.path.exists(fichier):
        urlretrieve(
            "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/data-science-avec-pandas/" + fichier,
            fichier,
        )

## Importer Pandas

In [ ]:
import pandas as pd

print(pd.__version__)

## Lire les fichiers CSV

In [ ]:
netflix_titles = pd.read_csv("titles.csv")
netflix_credits = pd.read_csv("credits.csv")

print(netflix_titles.shape)
print(netflix_credits.shape)

## Observer les données

Les 5 premières lignes des titres.

In [ ]:
pd.set_option("display.max_columns", 8)
netflix_titles.head()

Le générique : une ligne par personne et par titre.

In [ ]:
netflix_credits.head()

`info()` : types des colonnes et valeurs non manquantes.

In [ ]:
netflix_titles.info()

`describe()` : statistiques des colonnes numériques.

In [ ]:
netflix_titles[["release_year", "runtime", "imdb_score"]].describe().round(2)

## Sélectionner des colonnes et des lignes

Une colonne : le résultat est une Series.

In [ ]:
netflix_titles["description"]

Plusieurs colonnes : le résultat est un DataFrame.

In [ ]:
netflix_titles[["title", "description"]]

`loc` (étiquettes) et `iloc` (positions).

In [ ]:
print(netflix_titles.loc[0, "title"])
print(netflix_titles.iloc[2, 1])
netflix_titles.loc[:3, ["title", "type", "release_year"]]

## Filtrer

Une condition entre crochets.

In [ ]:
bien_notes = netflix_titles[netflix_titles["imdb_score"] > 8]

print(len(bien_notes), "titres avec une note IMDB supérieure à 8")
bien_notes[["title", "type", "release_year", "imdb_score"]].head()

Plusieurs conditions : `&`, `|`, `~`, avec des parenthèses.

In [ ]:
films_cultes = netflix_titles[
    (netflix_titles["type"] == "MOVIE") & (netflix_titles["imdb_score"] > 8.5)
]

print(len(films_cultes), "films avec une note supérieure à 8,5")
films_cultes[["title", "release_year", "imdb_score"]].sort_values("imdb_score", ascending=False).head()

`query()` : la même chose, de façon plus lisible.

In [ ]:
same = netflix_titles.query("type == 'MOVIE' and imdb_score > 8.5")

print(same.equals(films_cultes))

## Agréger avec groupby

Nombre de titres par type.

In [ ]:
netflix_titles.groupby(["type"]).agg(title_count=("title", "count"))

Plusieurs agrégats à la fois.

In [ ]:
netflix_titles.groupby("type").agg(
    title_count=("title", "count"),
    imdb_moyen=("imdb_score", "mean"),
    duree_mediane=("runtime", "median"),
).round(2)

Compter les genres : `ast.literal_eval` puis `explode()`.

In [ ]:
import ast

netflix_titles["genres"] = netflix_titles["genres"].apply(ast.literal_eval)   # texte -> vraie liste

genres = netflix_titles.explode("genres")
genres["genres"].value_counts().head(5)

## Valeurs manquantes

Compter les valeurs manquantes par colonne.

In [ ]:
netflix_titles.isna().sum()

`fillna()` : un film n'a pas de saison, donc 0.

In [ ]:
netflix_titles["seasons"] = netflix_titles["seasons"].fillna(0)

print(netflix_titles["seasons"].isna().sum(), "valeurs manquantes dans seasons")
print(netflix_titles.groupby("type")["seasons"].agg(["min", "max"]))

`dropna(subset=...)` : supprimer les lignes sans note IMDB.

In [ ]:
avec_note = netflix_titles.dropna(subset=["imdb_score"])

print(len(netflix_titles), "titres avant,", len(avec_note), "après dropna sur imdb_score")

## Trier

Ordre croissant.

In [ ]:
# trier les données en fonction de l'année de publication
netflix_titles.sort_values("release_year")[["title", "type", "release_year"]].head()

Plusieurs colonnes, avec des ordres différents.

In [ ]:
# année de sortie puis note IMDB, tous deux dans l'ordre décroissant
netflix_titles.sort_values(
    ["release_year", "imdb_score"], ascending=[False, False]
)[["title", "release_year", "imdb_score"]].head()

## Fusionner avec merge

Inner join : seulement les titres présents dans les deux tables.

In [ ]:
# fusionner les DataFrames sur la base de la colonne "id"
joined_df = pd.merge(netflix_titles, netflix_credits, on="id", how="inner")

print(joined_df.shape)
joined_df[["title", "type", "name", "role"]].head()

Left join avec `indicator=True` : on garde aussi les titres sans générique.

In [ ]:
avec_indicateur = pd.merge(
    netflix_titles, netflix_credits, on="id", how="left", indicator=True
)

sans_generique = avec_indicateur[avec_indicateur["_merge"] == "left_only"]
print(len(sans_generique), "titres sans aucune ligne dans credits")
print(len(avec_indicateur), "lignes avec un left join, contre", len(joined_df), "avec un inner join")

Attention : après la fusion, un titre apparaît autant de fois qu'il a de personnes au générique.

In [ ]:
credits_par_titre = netflix_credits.groupby("id").size()

print(credits_par_titre.describe().round(1))
print(joined_df["id"].nunique(), "titres distincts dans joined_df")

## Appliquer une fonction

`apply()` avec une lambda, comparé à l'opération vectorisée.

In [ ]:
# convertir la durée en heures avec apply et lambda
joined_df["runtime_hours"] = joined_df["runtime"].apply(lambda x: x / 60)

# la même opération, vectorisée
runtime_hours_vec = joined_df["runtime"] / 60

print(joined_df["runtime_hours"].equals(runtime_hours_vec))
joined_df[["title", "runtime", "runtime_hours"]].head()

## Visualiser

Histogramme du nombre de personnes au générique par titre.

In [ ]:
import matplotlib.pyplot as plt

role_count = joined_df.groupby("id").agg(count_role=("role", "count")).reset_index()

plt.hist(role_count["count_role"], bins=15, color="#2a78d6")
plt.xlabel("Nombre d'acteurs et de réalisateurs")
plt.ylabel("Nombre de titres")
plt.title("Combien de personnes figurent au générique d'un titre ?", loc="left")
plt.show()

print(role_count["count_role"].median(), "personnes au générique pour un titre médian")

Nombre de films et de séries par année de sortie. L'année 2023 est incomplète dans ce jeu de données.

In [ ]:
par_annee = (
    netflix_titles[netflix_titles["release_year"] >= 2000]
    .groupby(["release_year", "type"])
    .size()
    .unstack("type")
)

ax = par_annee.plot(color={"MOVIE": "#2a78d6", "SHOW": "#eb6834"}, figsize=(8, 4.5))
ax.set_xlabel("Année de sortie")
ax.set_ylabel("Nombre de titres")
ax.set_title("Films et séries au catalogue, selon l'année de sortie", loc="left")
ax.legend(["Films", "Séries"], frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

print(par_annee.loc[[2000, 2010, 2020]])

## Dates

Convertir l'année en date, l'utiliser comme index, puis sélectionner une période.

In [ ]:
titres_dates = netflix_titles.assign(
    release_date=pd.to_datetime(netflix_titles["release_year"].astype(str), format="%Y")
).set_index("release_date").sort_index()

print(titres_dates.index.dtype)
titres_dates.loc["2018":"2020", ["title", "type"]].head()

`resample()` : regrouper par tranches de 5 ans.

In [ ]:
# nombre de titres par période de 5 ans
titres_dates["id"].resample("5YS").count().tail(6)

Le même principe sur des dates quotidiennes.

In [ ]:
dates = pd.date_range("2023-01-01", periods=10)
df = pd.DataFrame({"A": range(1, 11), "B": range(11, 21)}, index=dates)

print(df.loc["2023-01-03":"2023-01-07"])
print(df.resample("W").mean())